The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [1]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [3]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
vocab = [word for word, _ in counts.most_common(V)]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
corpus = [word2idx[t] for t in tokens if t in word2idx]

## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [4]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        x = self.center(center_ids)
        return self.output(x)

In [6]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    np.random.shuffle(pairs)
    total_loss = 0.0
    for i in range(0, len(pairs), B):
        batch = pairs[i : i+B]
        center_ids = torch.tensor(batch[:, 0], dtype=torch.long)
        context_ids = torch.tensor(batch[:, 1], dtype=torch.long)
        
        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        total_loss += loss.item() * len(batch)
        
    print(f"Epoch {epoch + 1}/{epochs} loss={total_loss / len(pairs):.4f}")

emb = model.center.weight.detach().cpu().numpy()

Epoch 1/3 loss=6.9860
Epoch 2/3 loss=6.7127
Epoch 3/3 loss=6.5751


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [8]:
from sklearn.decomposition import PCA
from umap import UMAP

N = 1500
plot_words = vocab[:N]
X = emb[:N]

pca3 = PCA(n_components=3).fit_transform(X)

try:
  umap3 = UMAP(n_components=3, random_state=0).fit_transform(X)
except Exception:
  umap3 = None


/home/beninod34/cs375/eng-ai-agents/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [9]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = neighbor_set or set()
    colors, sizes = [], []
    for w in words:
        if w == query:
            colors.append("red")
            sizes.append(8)
        elif w in neighbor_set:
            colors.append("orange")
            sizes.append(6)
        else:
            colors.append("steelblue")
            sizes.append(3)

    fig = go.Figure(go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        text=words,
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=sizes, color=colors, opacity=0.7),
    ))
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))
    return fig

plot_embeddings(pca3, plot_words)

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [10]:
def neighbors(word, k=10):
    idx = word2idx[word]
    vec = emb[idx]
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    sims = (emb @ vec) / (norms.squeeze() * np.linalg.norm(vec))
    top = np.argsort(sims)[::-1]
    results = []
    for i in top:
        if i != idx:
            results.append((idx2word[i], float(sims[i])))
        if len(results) == k:
            break
    return results

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

federal         0.815
troops          0.814
municipal       0.797
authorities     0.793
pakistani       0.792
extending       0.789
subcontinent    0.788
courts          0.787
commonwealth    0.787
revolutionary   0.786


In [11]:
query = "government"
nbrs = neighbors(query, 10)
neighbor_set = {w for w, _ in nbrs}

plot_embeddings(pca3, plot_words, query=query, neighbor_set=neighbor_set)

## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

1.) Common nouns like "government", "city", and "war" return clean neighbors that are clearly related in meaning. A verb like "said" also works well and returns other speech verbs. A function word like "the" returns other function words like "a" and "an" which are grammatically similar but not semantically interesting. Rare words give noisy neighbors because the model sees them in very few contexts during training, so it does not have enough examples to learn a reliable position in the embedding space.

2.) Yes, related words do land near each other. In the PCA plot you can spot a cluster of country and place names, a cluster of number words, and a cluster of common function words all grouped together. The UMAP plot makes these groupings tighter and easier to see. Words that appear in similar sentences end up close together because the model learned their meaning from shared context.